In [3]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy import text

DB_USER = "postgres"
DB_PASSWORD = "zxcZXC#123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "retail-demand-pricing"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
print("Engine initialized successfully!")

Engine initialized successfully!


In [4]:
query = """
SELECT *
FROM retail.fact_demand
ORDER BY product_id, region, date;
"""

df = pd.read_sql(query, engine)
df["date"] = pd.to_datetime(df["date"])

print("Data loaded successfully. Shape:", df.shape)

Data loaded successfully. Shape: (50000, 47)


Calander Feature

In [5]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["week"] = df["date"].dt.isocalendar().week.astype(int)

Lag Feature

In [6]:
df = df.sort_values(
    ["product_id", "region", "date"]
)

g = df.groupby(
    ["product_id", "region"]
)["demand_units"]

for k in [1, 2, 4, 52]:
    df[f"lag_{k}"] = g.shift(k)

Rolling Feature

In [7]:
for w in [4, 13]:

    df[f"roll_mean_{w}"] = (
        g.transform(
            lambda s:
            s.shift(1)
             .rolling(w)
             .mean()
        )
    )

Price Relationship

In [8]:
df["relative_price"] = (
    df["selling_price"] /
    df["competitor_price"]
)

df["price_gap_pct"] = (
    (df["selling_price"] -
     df["competitor_price"])
    / df["competitor_price"]
    * 100
)

Removing rows where lag feature doesn't exist 

In [9]:
feature_cols = [
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_52",
    "roll_mean_4",
    "roll_mean_13"
]

df_model = df.dropna(
    subset=feature_cols
).copy()

In [10]:
with engine.begin() as connection:
    df_model.to_sql(
        "fact_demand_features",
        connection,
        schema="retail",
        if_exists="replace", 
        index=False
    )

print("Features written to PostgreSQL table: retail.fact_demand_features")

Features written to PostgreSQL table: retail.fact_demand_features
